# CROssBAR — Knowledge Graph Processing

All CROssBAR node/edge files are read from `CROSSBAR_PATH`.  
All reference/database files are read from `DB_BASE_PATH`.  
All output KG-triple CSVs are written to `OUT_PATH`.


## 0. Path Configuration

In [31]:
import os

# ── Edit only these four lines ───────────────────────────────────────────────
BASE_PATH     = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"      # general data collection root
DB_BASE_PATH  = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/databases_for_mapping"   # shared reference / database files
CROSSBAR_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/crossbar/"  # CROssBAR node & edge CSV/TSV files
OUT_PATH      = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/crossbar/"       # all output KG CSVs land here
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(OUT_PATH, exist_ok=True)
print("CROSSBAR_PATH :", CROSSBAR_PATH)
print("DB_BASE_PATH  :", DB_BASE_PATH)
print("OUT_PATH      :", OUT_PATH)


CROSSBAR_PATH : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/crossbar/
DB_BASE_PATH  : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/databases_for_mapping
OUT_PATH      : /storage/Arushi/090526_EvoAge/kg_formation/processed_data/crossbar/


## 1. Imports

In [32]:
import os
import numpy as np
import pandas as pd
from glob import glob


## 2. Shared Helpers


In [33]:
KG_DESIRED_COLS = [
    "Head", "Relation", "Tail",
    "Head_type", "Tail_type",
    "KG_Source",
    "Head_Alt_ID", "Head_Detail_name",
    "Tail_Alt_ID", "Tail_Detail_name",
    "Head_ID_IS", "Tail_ID_IS",
]

def reorder(df, extra_cols=None):
    """Return df keeping only KG_DESIRED_COLS (+ any extras) that exist, in order."""
    cols = KG_DESIRED_COLS + (extra_cols or [])
    return df[[c for c in cols if c in df.columns]]

GO_NS_MAP = {
    "biological_process": "BiologicalProcess",
    "molecular_function": "MolecularFunction",
    "cellular_component": "CellularComponent",
}


## 3. Reference Dictionaries

### 3.1 ChEMBL ID → Canonical SMILES

In [34]:
CHEMBL = pd.read_csv(
    os.path.join(DB_BASE_PATH, "CHEMBL", "chembl_35_chemreps.txt"), sep="\t"
)
CHEMBL_ID_SMILE = dict(zip(CHEMBL["chembl_id"], CHEMBL["canonical_smiles"]))
del CHEMBL
print(f"CHEMBL_ID_SMILE : {len(CHEMBL_ID_SMILE):,} entries")


CHEMBL_ID_SMILE : 2,474,590 entries


### 3.2 PubChem SMILES → CID

In [35]:
Pubchem = pd.read_pickle(os.path.join(DB_BASE_PATH, "pubchem", "combined_df.pkl"))
Pubchem_Smile_CID_Dict = dict(zip(Pubchem["PUBCHEM_SMILES"], Pubchem["PUBCHEM_COMPOUND_CID"]))
del Pubchem
print(f"Pubchem_Smile_CID_Dict : {len(Pubchem_Smile_CID_Dict):,} entries")


Pubchem_Smile_CID_Dict : 119,125,801 entries


### 3.3 DrugBank / ChEBI → PubChem CID

In [37]:
pubchem_raw  = pd.read_csv(os.path.join(DB_BASE_PATH,"pubchem", "Pubchem_ID_DrugBank_Chebi.csv"))
pubchem_db   = pubchem_raw[~pubchem_raw["DRUGBANK_ID"].isna()]
pubchem_chebi= pubchem_raw[~pubchem_raw["CHEBI_ID"].isna()]

DB2PC_dict    = dict(zip(pubchem_db["DRUGBANK_ID"],    pubchem_db["ID"]))
Chebi2PC_dict = dict(zip(pubchem_chebi["CHEBI_ID"],    pubchem_chebi["ID"]))
del pubchem_raw, pubchem_db, pubchem_chebi
print(f"DB2PC_dict    : {len(DB2PC_dict):,}")
print(f"Chebi2PC_dict : {len(Chebi2PC_dict):,}")


/tmp/ipykernel_1728732/2549068015.py:1: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  pubchem_raw  = pd.read_csv(os.path.join(DB_BASE_PATH,"pubchem", "Pubchem_ID_DrugBank_Chebi.csv"))


DB2PC_dict    : 10,702
Chebi2PC_dict : 177,680


### 3.4 HPO Phenotype ID → Name

In [38]:
phenotype = pd.read_csv(os.path.join(DB_BASE_PATH, "hpo", "HPO.csv"))
phenotype = phenotype[phenotype["id"].str.startswith("HP")]
phenotype_dict = dict(zip(phenotype["id"], phenotype["name"]))
del phenotype
print(f"phenotype_dict : {len(phenotype_dict):,}")


phenotype_dict : 19,483


### 3.5 GO ID → Name / Namespace

In [39]:
GO = pd.read_csv(os.path.join(DB_BASE_PATH, "MESH", "MESH_GO_ID_NAME.csv"))
GO_dict              = dict(zip(GO["id"], GO["name"]))
GO_id_namespace_dict = dict(zip(GO["id"], GO["namespace"]))
del GO
print(f"GO_dict : {len(GO_dict):,}")


GO_dict : 47,995


### 3.6 UniProt AC → RecName

In [44]:
Uniprot_Recname = pd.read_csv(os.path.join(DB_BASE_PATH, "uniprot", "Uniprot_ID_Recname.csv"))
Uniprot_Recname_dict = dict(zip(Uniprot_Recname["AC"], Uniprot_Recname["RecName"]))
del Uniprot_Recname
print(f"Uniprot_Recname_dict : {len(Uniprot_Recname_dict):,}")


Uniprot_Recname_dict : 794,151


In [51]:
!find ../ -name Homo_sapiens.gene_info

../data_collection/databases_for_mapping/ncbi/Homo_sapiens.gene_info


### 3.7 NCBI Gene dictionaries

In [52]:
ENS_2NCBI = pd.read_csv(os.path.join(DB_BASE_PATH, "ENSEMBL", "ENSEMBLE_ID_2_NCBI_Gene_SYM.csv"))
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI["name"].isna()]
NCBI_2ENS__dict = dict(zip(ENS_2NCBI["initial_alias"], ENS_2NCBI["name"]))
del ENS_2NCBI

NCBI_ALL_GENE = pd.read_csv(
    os.path.join(DB_BASE_PATH, "ncbi", "Homo_sapiens.gene_info"), sep="\t"
)
NCBI_ALL_GENE["ENSEMBLE_ID"]    = NCBI_ALL_GENE["Symbol"].map(NCBI_2ENS__dict)
NCBI_ALL_GENEname_dict          = dict(zip(NCBI_ALL_GENE["Symbol"],  NCBI_ALL_GENE["description"]))
NCBI_ALL_GENEIDname_dict        = dict(zip(NCBI_ALL_GENE["GeneID"],  NCBI_ALL_GENE["Symbol"]))
del NCBI_ALL_GENE
print(f"NCBI_ALL_GENEname_dict : {len(NCBI_ALL_GENEname_dict):,}")


NCBI_ALL_GENEname_dict : 193,352


### 3.8 Disease Ontology (DOID)

In [53]:
DO_All_File = pd.read_csv(os.path.join(DB_BASE_PATH, "DO", "All_DO_ref_files.csv"))
DOID_disname_dict      = dict(zip(DO_All_File["id"],    DO_All_File["label"]))
DOID_disAltID_dict     = dict(zip(DO_All_File["id"],    DO_All_File["xrefs"]))
DOID_disname_DOID_dict = dict(zip(DO_All_File["label"], DO_All_File["id"]))
del DO_All_File
print(f"DOID_disname_DOID_dict : {len(DOID_disname_DOID_dict):,}")


DOID_disname_DOID_dict : 11,787


### 3.9 Reactome Pathway Name → R-HSA ID

In [56]:
pathway_reactome = pd.read_csv(
    os.path.join(DB_BASE_PATH, "reactome", "ReactomePathways.txt"),
    sep="\t", header=None
)
pathway_reactome = pathway_reactome[pathway_reactome[0].str.startswith("R-HSA")]
pathway_reactome_dict = dict(zip(pathway_reactome[1], pathway_reactome[0]))  # name → R-HSA
del pathway_reactome
print(f"pathway_reactome_dict : {len(pathway_reactome_dict):,}")


pathway_reactome_dict : 2,734


## 4. Load CROssBAR Nodes & Edges

In [73]:
# !unzip /storage/Arushi/090526_EvoAge/kg_formation/data_collection/crossbar/CROssBAR-master/CROssBAR_INTEGRATED_KGs/CROssBAR-extended-KG-nodes.zip \
# -d /storage/Arushi/090526_EvoAge/kg_formation/data_collection/crossbar/CROssBAR-master/CROssBAR_INTEGRATED_KGs/

Archive:  /storage/Arushi/090526_EvoAge/kg_formation/data_collection/crossbar/CROssBAR-master/CROssBAR_INTEGRATED_KGs/CROssBAR-extended-KG-nodes.zip
  inflating: /storage/Arushi/090526_EvoAge/kg_formation/data_collection/crossbar/CROssBAR-master/CROssBAR_INTEGRATED_KGs/CROssBAR-extended-KG-nodes.csv  


In [76]:
# !unzip /storage/Arushi/090526_EvoAge/kg_formation/data_collection/crossbar/CROssBAR-master/CROssBAR_INTEGRATED_KGs/CROssBAR-extended-KG-edges.zip \
# -d /storage/Arushi/090526_EvoAge/kg_formation/data_collection/crossbar/CROssBAR-master/CROssBAR_INTEGRATED_KGs/

Archive:  /storage/Arushi/090526_EvoAge/kg_formation/data_collection/crossbar/CROssBAR-master/CROssBAR_INTEGRATED_KGs/CROssBAR-extended-KG-edges.zip
  inflating: /storage/Arushi/090526_EvoAge/kg_formation/data_collection/crossbar/CROssBAR-master/CROssBAR_INTEGRATED_KGs/CROssBAR-extended-KG-edges.csv  


In [78]:
file_nodes = pd.read_csv(
    os.path.join(CROSSBAR_PATH, "CROssBAR-master", "CROssBAR_INTEGRATED_KGs",
                 "CROssBAR-extended-KG-nodes.csv")
)

Nodes_ID_Label_dict    = dict(zip(file_nodes["neo4j_id"], file_nodes["_labels"]))
Nodes_ID_name_dict     = dict(zip(file_nodes["neo4j_id"], file_nodes["node_id"]))
Nodes_ID_fullname_dict = dict(zip(file_nodes["neo4j_id"], file_nodes["node_name"]))

print(f"Nodes loaded: {len(file_nodes):,}")
print("Node types:", file_nodes["_labels"].value_counts().to_dict())
del file_nodes


Nodes loaded: 873,568
Node types: {'Protein': 492574, 'Chebi': 160806, 'Compound': 135441, 'Go_term': 43435, 'Domain': 10229, 'Hpo': 9165, 'Drug': 6618, 'Ec_number': 5594, 'Disease': 3815, 'Pathway': 3764, 'Disease:Kegg': 1879, 'Kegg:Pathway': 248}


In [79]:
file_edges = pd.read_csv(
    os.path.join(CROSSBAR_PATH, "CROssBAR-master", "CROssBAR_INTEGRATED_KGs",
                 "CROssBAR-extended-KG-edges.csv")
)

file_edges["Head_type"]        = file_edges["source_neo4j_id"].map(Nodes_ID_Label_dict)
file_edges["Tail_type"]        = file_edges["target_neo4j_id"].map(Nodes_ID_Label_dict)
file_edges["Head_Name"]        = file_edges["source_neo4j_id"].map(Nodes_ID_name_dict)
file_edges["Tail_Name"]        = file_edges["target_neo4j_id"].map(Nodes_ID_name_dict)
file_edges["Head_Detail_name"] = file_edges["source_neo4j_id"].map(Nodes_ID_fullname_dict)
file_edges["Tail_Detail_name"] = file_edges["target_neo4j_id"].map(Nodes_ID_fullname_dict)

# Raw relation key (neo4j label pairs) — used only for filtering below
file_edges["Relation"] = file_edges["Head_type"] + "_" + file_edges["Tail_type"]
file_edges["KG_Source"] = "CROssBAR"
file_edges = file_edges.rename(columns={
    "source_neo4j_id": "Head",
    "target_neo4j_id": "Tail",
})

print(f"Edges loaded: {len(file_edges):,}")
print("\nRelation counts:")
print(file_edges["Relation"].value_counts().to_string())


Edges loaded: 5,258,348

Relation counts:
Relation
Protein_Go_term              3202059
Protein_Domain                731396
Chebi_Chebi                   319091
Protein_Ec_number             296781
Protein_Protein               294412
Compound_Protein              169363
Go_term_Go_term                85185
Hpo_Protein                    39888
Protein_Pathway                36534
Hpo_Disease                    22443
Protein_Kegg:Pathway           19474
Drug_Protein                   16727
Disease_Protein                 6626
Disease:Kegg_Protein            5901
Compound_Chebi                  3771
Drug_Chebi                      3038
Disease_Hpo                     2230
Disease:Kegg_Kegg:Pathway       1686
Protein_Disease                  683
Protein_Hpo                      442
Drug_Disease:Kegg                308
Protein_Compound                 259
Protein_Disease:Kegg              39
Protein_Drug                      12


## 5. Section Processing

Each section:
1. Filters `file_edges` by raw `Relation`
2. Resolves IDs to canonical namespaces
3. Sets `Head_type`, `Tail_type`, `Relation`, `Head_ID_IS`, `Tail_ID_IS`
4. Calls `reorder()` and saves to `OUT_PATH`

### §1 Protein → GO Term (BiologicalProcess / MolecularFunction / CellularComponent)


In [80]:
Protein_Go_term = file_edges[file_edges["Relation"] == "Protein_Go_term"].copy()

Protein_Go_term["Tail_GO_Type"]    = Protein_Go_term["Tail_Name"].map(GO_id_namespace_dict)
Protein_Go_term["Tail_Detail_name"]= Protein_Go_term["Tail_Name"].map(GO_dict)
Protein_Go_term = Protein_Go_term[~Protein_Go_term["Tail_GO_Type"].isna()]

Protein_Go_term["Head_Detail_name"] = Protein_Go_term["Head_Name"].map(Uniprot_Recname_dict)
Protein_Go_term = Protein_Go_term[~Protein_Go_term["Head_Detail_name"].isna()]

# Swap neo4j IDs → canonical IDs
Protein_Go_term[["Head_Name", "Head"]] = Protein_Go_term[["Head", "Head_Name"]]
Protein_Go_term[["Tail_Name", "Tail"]] = Protein_Go_term[["Tail", "Tail_Name"]]


Protein_Go_term["Tail_type"] = Protein_Go_term["Tail_GO_Type"].map(GO_NS_MAP)
Protein_Go_term = Protein_Go_term[~Protein_Go_term["Tail_type"].isna()]

Protein_Go_term = Protein_Go_term.rename(columns={"Head_Detail_name": "Head_Alt_ID"})
Protein_Go_term.drop(columns=["Head_Name", "Tail_Name", "Tail_GO_Type"], inplace=True)

Protein_Go_term["Relation"]   = "Protein_" + Protein_Go_term["Tail_type"]
Protein_Go_term["Head_ID_IS"] = "Uniprot_ID"
Protein_Go_term["Tail_ID_IS"] = "QuickGO"

Protein_Go_term = reorder(Protein_Go_term)
print(f"Protein–GO rows: {len(Protein_Go_term):,}")
print(Protein_Go_term["Tail_type"].value_counts().to_string())


Protein–GO rows: 3,142,665
Tail_type
BiologicalProcess    1235325
MolecularFunction    1138818
CellularComponent     768522


In [81]:
for go_type in ["BiologicalProcess", "MolecularFunction", "CellularComponent"]:
    subset = Protein_Go_term[Protein_Go_term["Tail_type"] == go_type]
    fname  = f"Protein_{go_type}_Crossbar.csv"
    subset.to_csv(os.path.join(OUT_PATH, fname), index=False)
    print(f"Saved {fname}: {len(subset):,} rows")
del Protein_Go_term


Saved Protein_BiologicalProcess_Crossbar.csv: 1,235,325 rows
Saved Protein_MolecularFunction_Crossbar.csv: 1,138,818 rows
Saved Protein_CellularComponent_Crossbar.csv: 768,522 rows


### §2 ChEBI–ChEBI → Chemical–Chemical

In [87]:
Chebi_Chebi = file_edges[file_edges["Relation"] == "Chebi_Chebi"].copy()
Chebi_Chebi.drop(columns=["Head", "Tail"], inplace=True)

Chebi_Chebi["Head"] = Chebi_Chebi["Head_Name"].map(Chebi2PC_dict)
Chebi_Chebi["Tail"] = Chebi_Chebi["Tail_Name"].map(Chebi2PC_dict)
Chebi_Chebi = Chebi_Chebi[~Chebi_Chebi["Head"].isna() & ~Chebi_Chebi["Tail"].isna()]

Chebi_Chebi["Head"] = Chebi_Chebi["Head"].astype(str).str.replace(r"\.0$", "", regex=True)
Chebi_Chebi["Tail"] = Chebi_Chebi["Tail"].astype(str).str.replace(r"\.0$", "", regex=True)

Chebi_Chebi = Chebi_Chebi.rename(columns={
    "Head_Name": "Head_Alt_ID", "Tail_Name": "Tail_Alt_ID"
})
Chebi_Chebi["Relation"]   = "Chemical_Chemical"
Chebi_Chebi["Head_type"]  = "ChemicalEntity"
Chebi_Chebi["Tail_type"]  = "ChemicalEntity"
Chebi_Chebi["Head_ID_IS"] = "Pubchem"
Chebi_Chebi["Tail_ID_IS"] = "Pubchem"

Chebi_Chebi = reorder(Chebi_Chebi)
print(f"ChEBI–ChEBI rows: {len(Chebi_Chebi):,}")
Chebi_Chebi.to_csv(os.path.join(OUT_PATH, "ChemicalEntity_ChemicalEntity_Crossbar.csv"), index=False)
print("Saved: ChemicalEntity_ChemicalEntity_Crossbar.csv")
del Chebi_Chebi


ChEBI–ChEBI rows: 50,019
Saved: ChemicalEntity_ChemicalEntity_Crossbar.csv


### §3 Protein–Protein Interactions

In [88]:
Protein_Protein = file_edges[file_edges["Relation"] == "Protein_Protein"].copy()

Protein_Protein["Head_Detail_name"] = Protein_Protein["Head_Name"].map(Uniprot_Recname_dict)
Protein_Protein = Protein_Protein[~Protein_Protein["Head_Detail_name"].isna()]
Protein_Protein[["Head_Name", "Head"]] = Protein_Protein[["Head", "Head_Name"]]

Protein_Protein["Tail_Detail_name"] = Protein_Protein["Tail_Name"].map(Uniprot_Recname_dict)
Protein_Protein = Protein_Protein[~Protein_Protein["Tail_Detail_name"].isna()]
Protein_Protein[["Tail_Name", "Tail"]] = Protein_Protein[["Tail", "Tail_Name"]]

Protein_Protein.drop(columns=["Head_Name", "Tail_Name"], inplace=True)
Protein_Protein["Head_ID_IS"] = "Uniprot_ID"
Protein_Protein["Tail_ID_IS"] = "Uniprot_ID"

Protein_Protein = reorder(Protein_Protein)
print(f"Protein–Protein rows: {len(Protein_Protein):,}")
Protein_Protein.to_csv(os.path.join(OUT_PATH, "Protein_Protein_Crossbar.csv"), index=False)
print("Saved: Protein_Protein_Crossbar.csv")
del Protein_Protein


Protein–Protein rows: 292,762
Saved: Protein_Protein_Crossbar.csv


### §4 Compound–Protein (ChEMBL → PubChem CID via SMILES)

In [89]:
Compound_Protein = file_edges[file_edges["Relation"] == "Compound_Protein"].copy()

Compound_Protein["Head_Smiles"] = Compound_Protein["Head_Name"].map(CHEMBL_ID_SMILE)
Compound_Protein["Head"]        = Compound_Protein["Head_Smiles"].map(Pubchem_Smile_CID_Dict)
Compound_Protein = Compound_Protein[~Compound_Protein["Head"].isna()]

Compound_Protein[["Tail_Name", "Tail"]] = Compound_Protein[["Tail", "Tail_Name"]]

Compound_Protein = Compound_Protein.rename(columns={
    "Head_Name": "Head_Alt_ID", "Tail_Detail_name": "Tail_Alt_ID"
})
Compound_Protein.drop(columns=["Tail_Name", "Head_Detail_name", "Head_Smiles"], inplace=True)

Compound_Protein["Head_type"]  = "ChemicalEntity"
Compound_Protein["Relation"]   = "ChemicalEntity_Protein"
Compound_Protein["Head_ID_IS"] = "Pubchem"
Compound_Protein["Tail_ID_IS"] = "Uniprot_ID"

Compound_Protein = reorder(Compound_Protein)
print(f"Compound–Protein rows: {len(Compound_Protein):,}")
Compound_Protein.to_csv(os.path.join(OUT_PATH, "Chemical_Protein_Crossbar.csv"), index=False)
print("Saved: Chemical_Protein_Crossbar.csv")
del Compound_Protein


Compound–Protein rows: 568
Saved: Chemical_Protein_Crossbar.csv


### §5 GO Term–GO Term (BiologicalProcess / MolecularFunction / CellularComponent)

In [90]:
Go_term_Go_term = file_edges[file_edges["Relation"] == "Go_term_Go_term"].copy()

Go_term_Go_term["Head_GO_Type"] = Go_term_Go_term["Head_Name"].map(GO_id_namespace_dict)
Go_term_Go_term["Tail_GO_Type"] = Go_term_Go_term["Tail_Name"].map(GO_id_namespace_dict)

Go_term_Go_term[["Head_Name", "Head"]] = Go_term_Go_term[["Head", "Head_Name"]]
Go_term_Go_term[["Tail_Name", "Tail"]] = Go_term_Go_term[["Tail", "Tail_Name"]]

Go_term_Go_term.drop(columns=["Head_Name", "Tail_Name", "Head_type", "Tail_type"], inplace=True)

Go_term_Go_term["Head_type"] = Go_term_Go_term["Head_GO_Type"].map(GO_NS_MAP)
Go_term_Go_term["Tail_type"] = Go_term_Go_term["Tail_GO_Type"].map(GO_NS_MAP)
Go_term_Go_term.drop(columns=["Head_GO_Type", "Tail_GO_Type"], inplace=True)
Go_term_Go_term = Go_term_Go_term[
    ~Go_term_Go_term["Head_type"].isna() & ~Go_term_Go_term["Tail_type"].isna()
]

Go_term_Go_term["Relation"]   = Go_term_Go_term["Head_type"] + "_" + Go_term_Go_term["Tail_type"]
Go_term_Go_term["Head_ID_IS"] = "QuickGO"
Go_term_Go_term["Tail_ID_IS"] = "QuickGO"

Go_term_Go_term = reorder(Go_term_Go_term)
print(f"GO–GO rows: {len(Go_term_Go_term):,}")
print(Go_term_Go_term["Relation"].value_counts().to_string())


GO–GO rows: 85,154
Relation
BiologicalProcess_BiologicalProcess    64906
MolecularFunction_MolecularFunction    13760
CellularComponent_CellularComponent     6488


In [91]:
for rel in ["BiologicalProcess_BiologicalProcess",
            "MolecularFunction_MolecularFunction",
            "CellularComponent_CellularComponent"]:
    subset = Go_term_Go_term[Go_term_Go_term["Relation"] == rel]
    fname  = f"{rel}_Crossbar.csv"
    subset.to_csv(os.path.join(OUT_PATH, fname), index=False)
    print(f"Saved {fname}: {len(subset):,} rows")
del Go_term_Go_term


Saved BiologicalProcess_BiologicalProcess_Crossbar.csv: 64,906 rows
Saved MolecularFunction_MolecularFunction_Crossbar.csv: 13,760 rows
Saved CellularComponent_CellularComponent_Crossbar.csv: 6,488 rows


### §6 HPO–Protein → Phenotype–Protein

In [92]:
Hpo_Protein = file_edges[file_edges["Relation"] == "Hpo_Protein"].copy()

Hpo_Protein[["Head_Name", "Head"]] = Hpo_Protein[["Head", "Head_Name"]]
Hpo_Protein[["Tail_Name", "Tail"]] = Hpo_Protein[["Tail", "Tail_Name"]]

Hpo_Protein["Head_type"] = "Phenotype"
Hpo_Protein["Relation"]  = "Phenotype_Protein"

Hpo_Protein["Tail_Detail_name"] = Hpo_Protein["Tail"].map(Uniprot_Recname_dict)
Hpo_Protein = Hpo_Protein[~Hpo_Protein["Tail_Detail_name"].isna()]

Hpo_Protein = Hpo_Protein.rename(columns={"Tail_Alt_ID": "Tail_Alt_ID_orig",
                                            "Tail_Detail_name_orig": "x"})
Hpo_Protein.drop(columns=["Head_Name", "Tail_Name"], inplace=True)
Hpo_Protein = Hpo_Protein.rename(columns={"Head_Detail_name": "Head_Alt_ID"})

Hpo_Protein["Head_ID_IS"] = "HPO_ID"
Hpo_Protein["Tail_ID_IS"] = "Uniprot_ID"

Hpo_Protein = reorder(Hpo_Protein)
print(f"Phenotype–Protein rows: {len(Hpo_Protein):,}")
Hpo_Protein.to_csv(os.path.join(OUT_PATH, "Phenotype_Protein_Crossbar.csv"), index=False)
print("Saved: Phenotype_Protein_Crossbar.csv")
del Hpo_Protein


Phenotype–Protein rows: 39,720
Saved: Phenotype_Protein_Crossbar.csv


### §7 Protein–Pathway (Reactome)

In [93]:
Protein_Pathway = file_edges[file_edges["Relation"] == "Protein_Pathway"].copy()

Protein_Pathway[["Head_Name", "Head"]] = Protein_Pathway[["Head", "Head_Name"]]
Protein_Pathway[["Tail_Name", "Tail"]] = Protein_Pathway[["Tail", "Tail_Name"]]

Protein_Pathway["Head_Detail_name"] = Protein_Pathway["Head"].map(Uniprot_Recname_dict)
Protein_Pathway = Protein_Pathway[~Protein_Pathway["Head_Detail_name"].isna()]

Protein_Pathway.drop(columns=["Head_Name", "Tail_Name"], inplace=True)
Protein_Pathway["Head_ID_IS"] = "Uniprot_ID"
Protein_Pathway["Tail_ID_IS"] = "Reactome_ID"

Protein_Pathway = reorder(Protein_Pathway)
print(f"Protein–Pathway rows: {len(Protein_Pathway):,}")
Protein_Pathway.to_csv(os.path.join(OUT_PATH, "Protein_Pathway_Crossbar.csv"), index=False)
print("Saved: Protein_Pathway_Crossbar.csv")
del Protein_Pathway


Protein–Pathway rows: 36,496
Saved: Protein_Pathway_Crossbar.csv


### §8 HPO–Disease — **Skipped**
Not processed: no Orphanet → DOID mapping available.

### §9 Drug–Protein → ChemicalEntity–Protein


In [94]:
Drug_Protein = file_edges[file_edges["Relation"] == "Drug_Protein"].copy()

Drug_Protein[["Head_Name", "Head"]] = Drug_Protein[["Head", "Head_Name"]]
Drug_Protein[["Tail_Name", "Tail"]] = Drug_Protein[["Tail", "Tail_Name"]]


Drug_Protein["Head_PubChem"] = Drug_Protein["Head"].map(DB2PC_dict)
Drug_Protein["Head_PubChem"] = Drug_Protein["Head_PubChem"].astype(str).str.replace(r"\.0$", "", regex=True)
Drug_Protein = Drug_Protein[~Drug_Protein["Head_PubChem"].isna() & (Drug_Protein["Head_PubChem"] != "nan")]

Drug_Protein[["Head_Alt_ID", "Head"]] = Drug_Protein[["Head", "Head_PubChem"]]
Drug_Protein.drop(columns=["Head_Name", "Tail_Name", "Head_PubChem", "Head_Detail_name"], inplace=True)

Drug_Protein["Head_type"]  = "ChemicalEntity"
Drug_Protein["Relation"]   = "ChemicalEntity_Protein"
Drug_Protein["Head_ID_IS"] = "Pubchem"
Drug_Protein["Tail_ID_IS"] = "Uniprot_ID"

Drug_Protein = reorder(Drug_Protein)
print(f"Drug–Protein rows: {len(Drug_Protein):,}")
Drug_Protein.to_csv(os.path.join(OUT_PATH, "ChemicalEntity_Protein_Crossbar.csv"), index=False)
print("Saved: ChemicalEntity_Protein_Crossbar.csv")
del Drug_Protein


Drug–Protein rows: 15,201
Saved: ChemicalEntity_Protein_Crossbar.csv


### §10 Disease–Protein


In [95]:
Disease_Protein = file_edges[file_edges["Relation"] == "Disease_Protein"].copy()

Disease_Protein["DOID"] = Disease_Protein["Head_Detail_name"].map(DOID_disname_DOID_dict)
Disease_Protein = Disease_Protein[~Disease_Protein["DOID"].isna()]

Disease_Protein[["DOID", "Head"]] = Disease_Protein[["Head", "DOID"]]

Disease_Protein[["Tail", "Tail_Name"]] = Disease_Protein[["Tail_Name", "Tail"]]

Disease_Protein = Disease_Protein.rename(columns={
    "Head_Name": "Head_Alt_ID",
    "Tail_Detail_name": "Tail_Alt_ID",
})

Disease_Protein.drop(columns=["DOID"], inplace=True)

Disease_Protein["Head_ID_IS"] = "DOID"
Disease_Protein["Tail_ID_IS"] = "Uniprot_ID"

Disease_Protein = reorder(Disease_Protein)
print(f"Disease–Protein rows: {len(Disease_Protein):,}")
Disease_Protein.to_csv(os.path.join(OUT_PATH, "Disease_Protein_1_Crossbar.csv"), index=False)
print("Saved: Disease_Protein_1_Crossbar.csv")
del Disease_Protein


Disease–Protein rows: 1,229
Saved: Disease_Protein_1_Crossbar.csv


### §11 Disease:Kegg–Protein

In [96]:
Disease_Kegg_Protein = file_edges[file_edges["Relation"] == "Disease:Kegg_Protein"].copy()

Disease_Kegg_Protein[["Head_Name", "Head"]] = Disease_Kegg_Protein[["Head", "Head_Name"]]
Disease_Kegg_Protein[["Tail_Name", "Tail"]] = Disease_Kegg_Protein[["Tail", "Tail_Name"]]

Disease_Kegg_Protein["DOID"] = Disease_Kegg_Protein["Head_Detail_name"].map(DOID_disname_DOID_dict)
Disease_Kegg_Protein = Disease_Kegg_Protein[~Disease_Kegg_Protein["DOID"].isna()]

Disease_Kegg_Protein[["Head", "DOID"]] = Disease_Kegg_Protein[["DOID", "Head"]]
Disease_Kegg_Protein.drop(columns=["Head_Name", "Tail_Name", "DOID"], inplace=True)

Disease_Kegg_Protein = Disease_Kegg_Protein.rename(columns={"Tail_Detail_name": "Tail_Alt_ID"})

Disease_Kegg_Protein["Head_ID_IS"] = "DOID"
Disease_Kegg_Protein["Tail_ID_IS"] = "Uniprot_ID"

Disease_Kegg_Protein = reorder(Disease_Kegg_Protein)
print(f"Disease:Kegg–Protein rows: {len(Disease_Kegg_Protein):,}")
Disease_Kegg_Protein.to_csv(os.path.join(OUT_PATH, "Disease_Protein_Crossbar.csv"), index=False)
print("Saved: Disease_Protein_Crossbar.csv")
del Disease_Kegg_Protein


Disease:Kegg–Protein rows: 702
Saved: Disease_Protein_Crossbar.csv


### §12 Disease–HPO → Disease–Phenotype

In [97]:
Disease_Hpo = file_edges[file_edges["Relation"] == "Disease_Hpo"].copy()

Disease_Hpo[["Tail_Name", "Tail"]] = Disease_Hpo[["Tail", "Tail_Name"]]

Disease_Hpo["DOID"] = Disease_Hpo["Head_Detail_name"].map(DOID_disname_DOID_dict)
Disease_Hpo = Disease_Hpo[~Disease_Hpo["DOID"].isna()]
Disease_Hpo[["DOID", "Head"]] = Disease_Hpo[["Head", "DOID"]]

Disease_Hpo["Relation"]  = "Disease_Phenotype"
Disease_Hpo["Tail_type"] = "Phenotype"

Disease_Hpo = Disease_Hpo.rename(columns={"Head_Name": "Head_Alt_ID"})
Disease_Hpo.drop(columns=["Tail_Name", "DOID"], inplace=True)

Disease_Hpo["Head_ID_IS"] = "DOID"
Disease_Hpo["Tail_ID_IS"] = "HPO_ID"

Disease_Hpo = reorder(Disease_Hpo)
print(f"Disease–Phenotype rows: {len(Disease_Hpo):,}")
Disease_Hpo.to_csv(os.path.join(OUT_PATH, "Disease_Phenotype_Crossbar.csv"), index=False)
print("Saved: Disease_Phenotype_Crossbar.csv")
del Disease_Hpo


Disease–Phenotype rows: 920
Saved: Disease_Phenotype_Crossbar.csv


### §13 Disease:Kegg–Kegg:Pathway — **Skipped**
Commented out in source; kept here for completeness.

### §14 Protein–Disease

In [98]:
Protein_Disease = file_edges[file_edges["Relation"] == "Protein_Disease"].copy()

Protein_Disease["DOID"] = Protein_Disease["Tail_Detail_name"].map(DOID_disname_DOID_dict)
Protein_Disease = Protein_Disease[~Protein_Disease["DOID"].isna()]

Protein_Disease[["DOID", "Tail"]]      = Protein_Disease[["Tail", "DOID"]]
Protein_Disease[["Head_Name", "Head"]] = Protein_Disease[["Head", "Head_Name"]]

Protein_Disease = Protein_Disease.rename(columns={
    "Head_Detail_name": "Head_Alt_ID",
    "Tail_Name": "Tail_Alt_ID",
})
Protein_Disease.drop(columns=["DOID", "Head_Name"], inplace=True)

Protein_Disease["Head_ID_IS"] = "Uniprot_ID"
Protein_Disease["Tail_ID_IS"] = "DOID"

Protein_Disease = reorder(Protein_Disease)
print(f"Protein–Disease rows: {len(Protein_Disease):,}")
Protein_Disease.to_csv(os.path.join(OUT_PATH, "Protein_Disease_Crossbar.csv"), index=False)
print("Saved: Protein_Disease_Crossbar.csv")
del Protein_Disease


Protein–Disease rows: 92
Saved: Protein_Disease_Crossbar.csv


### §15 Protein–HPO → Protein–Phenotype

In [99]:
Protein_Hpo = file_edges[file_edges["Relation"] == "Protein_Hpo"].copy()

Protein_Hpo[["Tail_Name", "Tail"]] = Protein_Hpo[["Tail", "Tail_Name"]]
Protein_Hpo[["Head_Name", "Head"]] = Protein_Hpo[["Head", "Head_Name"]]

Protein_Hpo.drop(columns=["Head_Name", "Tail_Name"], inplace=True)
Protein_Hpo = Protein_Hpo.rename(columns={"Head_Detail_name": "Head_Alt_ID"})

Protein_Hpo["Head_Detail_name"] = Protein_Hpo["Head"].map(Uniprot_Recname_dict)
Protein_Hpo = Protein_Hpo[~Protein_Hpo["Head_Detail_name"].isna()]

Protein_Hpo["Head_ID_IS"] = "Uniprot_ID"
Protein_Hpo["Tail_ID_IS"] = "HPO_ID"
Protein_Hpo["Tail_type"]  = "Phenotype"
Protein_Hpo["Relation"]   = "Protein_Phenotype"

Protein_Hpo = reorder(Protein_Hpo)
print(f"Protein–Phenotype rows: {len(Protein_Hpo):,}")
Protein_Hpo.to_csv(os.path.join(OUT_PATH, "Protein_Phenotype_Crossbar.csv"), index=False)
print("Saved: Protein_Phenotype_Crossbar.csv")
del Protein_Hpo


Protein–Phenotype rows: 328
Saved: Protein_Phenotype_Crossbar.csv


### §16 Drug–Disease:Kegg → ChemicalEntity–Disease


In [101]:
Drug_Disease_Kegg = file_edges[file_edges["Relation"] == "Drug_Disease:Kegg"].copy()

Drug_Disease_Kegg["Head"] = (
    Drug_Disease_Kegg["Head_Name"].map(DB2PC_dict)
    .fillna(Drug_Disease_Kegg["Head_Name"])
)
Drug_Disease_Kegg["Head"] = Drug_Disease_Kegg["Head"].astype(str).str.replace(r"\.0$", "", regex=True)


Drug_Disease_Kegg["DOID"] = Drug_Disease_Kegg["Tail_Detail_name"].map(DOID_disname_DOID_dict)
Drug_Disease_Kegg = Drug_Disease_Kegg[~Drug_Disease_Kegg["DOID"].isna()]
Drug_Disease_Kegg[["DOID", "Tail"]] = Drug_Disease_Kegg[["Tail", "DOID"]]

# Drug_Disease_Kegg["Head_ID_IS"] = np.where(
#     Drug_Disease_Kegg["Head"].isna(), np.nan,
#     np.where(Drug_Disease_Kegg["Head"].astype(str).str.startswith("DB"), "Drugbank", "Pubchem")
# )

Drug_Disease_Kegg["Head_ID_IS"] = Drug_Disease_Kegg["Head"].apply(lambda x: np.nan if pd.isna(x) else ("Drugbank" if str(x).startswith("DB") else "Pubchem"))

Drug_Disease_Kegg["Tail_ID_IS"] = "DOID"
Drug_Disease_Kegg["Head_type"]  = "ChemicalEntity"
Drug_Disease_Kegg["Tail_type"]  = "Disease"
Drug_Disease_Kegg["Relation"]   = "ChemicalEntity_Disease"

Drug_Disease_Kegg = Drug_Disease_Kegg.rename(columns={
    "Head_Name": "Head_Alt_ID", "Tail_Name": "Tail_Alt_ID"
})
Drug_Disease_Kegg.drop(columns=["DOID"], inplace=True)

Drug_Disease_Kegg = reorder(Drug_Disease_Kegg)
print(f"Drug–Disease rows: {len(Drug_Disease_Kegg):,}")
Drug_Disease_Kegg.to_csv(os.path.join(OUT_PATH, "ChemicalEntity_Disease_Crossbar.csv"), index=False)
print("Saved: ChemicalEntity_Disease_Crossbar.csv")
del Drug_Disease_Kegg



Drug–Disease rows: 6
Saved: ChemicalEntity_Disease_Crossbar.csv


### §17 Protein–Compound — **Skipped**
Commented out in source; kept here for completeness.

### §18 Protein–Disease:Kegg → Protein–Disease


In [102]:
Protein_DiseaseKegg = file_edges[file_edges["Relation"] == "Protein_Disease:Kegg"].copy()

Protein_DiseaseKegg[["Head_Name", "Head"]] = Protein_DiseaseKegg[["Head", "Head_Name"]]
Protein_DiseaseKegg.drop(columns=["Head_Name"], inplace=True)

Protein_DiseaseKegg = Protein_DiseaseKegg.rename(columns={"Head_Detail_name": "Head_Alt_ID"})
Protein_DiseaseKegg["Head_Detail_name"] = Protein_DiseaseKegg["Head"].map(Uniprot_Recname_dict)


Protein_DiseaseKegg["DOID"] = Protein_DiseaseKegg["Tail_Detail_name"].map(DOID_disname_DOID_dict)
Protein_DiseaseKegg = Protein_DiseaseKegg[~Protein_DiseaseKegg["DOID"].isna()]
Protein_DiseaseKegg[["DOID", "Tail"]] = Protein_DiseaseKegg[["Tail", "DOID"]]
Protein_DiseaseKegg.drop(columns=["DOID"], inplace=True)

Protein_DiseaseKegg["Tail_type"] = "Disease"
Protein_DiseaseKegg["Relation"]  = "Protein_Disease"
Protein_DiseaseKegg["Head_ID_IS"] = "Uniprot_ID"
Protein_DiseaseKegg["Tail_ID_IS"] = "DOID"

Protein_DiseaseKegg = reorder(Protein_DiseaseKegg)
print(f"Protein–Disease:Kegg rows: {len(Protein_DiseaseKegg):,}")
Protein_DiseaseKegg.to_csv(os.path.join(OUT_PATH, "Protein_Disease_2_Crossbar.csv"), index=False)
print("Saved: Protein_Disease_2_Crossbar.csv")
del Protein_DiseaseKegg


Protein–Disease:Kegg rows: 3
Saved: Protein_Disease_2_Crossbar.csv


### §19 Protein–Drug → Protein–ChemicalEntity

In [103]:
Protein_Drug = file_edges[file_edges["Relation"] == "Protein_Drug"].copy()

Protein_Drug["Head"] = Protein_Drug["Head_Name"].map(Uniprot_Recname_dict)
Protein_Drug[["Head_Name", "Head"]] = Protein_Drug[["Head", "Head_Name"]]
Protein_Drug[["Head_Name", "Head_Detail_name"]] = Protein_Drug[["Head_Detail_name", "Head_Name"]]

Protein_Drug["Tail"] = (
    Protein_Drug["Tail_Name"].map(DB2PC_dict)
    .fillna(Protein_Drug["Tail_Name"])
)
Protein_Drug["Tail"] = Protein_Drug["Tail"].astype(str).str.replace(r"\.0$", "", regex=True)

Protein_Drug = Protein_Drug.rename(columns={
    "Head_Name": "Head_Alt_ID", "Tail_Name": "Tail_Alt_ID"
})
Protein_Drug["Tail_type"] = "ChemicalEntity"
Protein_Drug["Relation"]  = "Protein_ChemicalEntity"
Protein_Drug["Head_ID_IS"] = "Uniprot_ID"
Protein_Drug["Tail_ID_IS"] = "Pubchem"

Protein_Drug = reorder(Protein_Drug)
print(f"Protein–Drug rows: {len(Protein_Drug):,}")
Protein_Drug.to_csv(os.path.join(OUT_PATH, "Protein_ChemicalEntity_Crossbar.csv"), index=False)
print("Saved: Protein_ChemicalEntity_Crossbar.csv")
del Protein_Drug


Protein–Drug rows: 12
Saved: Protein_ChemicalEntity_Crossbar.csv


### §20 Protein–Kegg:Pathway → Protein–Pathway


In [104]:
Protein_KeggPathway = file_edges[file_edges["Relation"] == "Protein_Kegg:Pathway"].copy()

Protein_KeggPathway[["Head_Name", "Head"]] = Protein_KeggPathway[["Head", "Head_Name"]]
Protein_KeggPathway[["Tail_Name", "Tail"]] = Protein_KeggPathway[["Tail", "Tail_Name"]]

Protein_KeggPathway["Tail_React_ID"] = (
    Protein_KeggPathway["Tail_Detail_name"].map(pathway_reactome_dict)
    .fillna(Protein_KeggPathway["Tail"])
)
Protein_KeggPathway[["Tail_React_ID", "Tail"]] = Protein_KeggPathway[["Tail", "Tail_React_ID"]]
Protein_KeggPathway = Protein_KeggPathway.rename(columns={"Tail_React_ID": "Tail_Alt_ID"})
Protein_KeggPathway.drop(columns=["Head_Name", "Tail_Name"], inplace=True)

Protein_KeggPathway["Tail_type"] = "Pathway"
Protein_KeggPathway["Relation"]  = "Protein_Pathway"
Protein_KeggPathway["Head_ID_IS"] = "Uniprot_ID"


Protein_KeggPathway["Tail_ID_IS"] = (
    Protein_KeggPathway["Tail"]
    .fillna("")
    .apply(lambda x: "KEGG_ID" if x.startswith("hsa") else ("Reactome_ID" if x else np.nan))
)

Protein_KeggPathway = reorder(Protein_KeggPathway)
print(f"Protein–Kegg:Pathway rows: {len(Protein_KeggPathway):,}")
print(Protein_KeggPathway["Tail_ID_IS"].value_counts().to_string())
Protein_KeggPathway.to_csv(os.path.join(OUT_PATH, "Protein_Pathway_Crossbar_2.csv"), index=False)
print("Saved: Protein_Pathway_Crossbar_2.csv")
del Protein_KeggPathway


Protein–Kegg:Pathway rows: 19,474
Tail_ID_IS
KEGG_ID        18542
Reactome_ID      932
Saved: Protein_Pathway_Crossbar_2.csv


## §21 Summary Audit

In [105]:
cols_audit = ["Relation", "Head_type", "Tail_type", "KG_Source", "Head_ID_IS", "Tail_ID_IS"]
rows = []
for fp in sorted(glob(os.path.join(OUT_PATH, "*.csv"))):
    try:
        tmp = pd.read_csv(fp)
        row = {"File": os.path.basename(fp), "Triples": len(tmp)}
        for col in cols_audit:
            if col in tmp.columns:
                row[col] = set(tmp[col].dropna().unique())
        rows.append(row)
    except Exception as e:
        print(f"  Error: {fp} — {e}")

audit_df = pd.concat([pd.DataFrame([r]) for r in rows], ignore_index=True)
display(audit_df)
print(f"\nTotal triples: {audit_df['Triples'].sum():,}")


,File,Triples,Relation,Head_type,Tail_type,KG_Source,Head_ID_IS,Tail_ID_IS
0,BiologicalProcess_BiologicalProcess_Crossbar.csv,64906,{BiologicalProcess_BiologicalProcess},{BiologicalProcess},{BiologicalProcess},{CROssBAR},{QuickGO},{QuickGO}
1,CellularComponent_CellularComponent_Crossbar.csv,6488,{CellularComponent_CellularComponent},{CellularComponent},{CellularComponent},{CROssBAR},{QuickGO},{QuickGO}
2,ChemicalEntity_ChemicalEntity_Crossbar.csv,50019,{Chemical_Chemical},{ChemicalEntity},{ChemicalEntity},{CROssBAR},{Pubchem},{Pubchem}
3,ChemicalEntity_Disease_Crossbar.csv,6,{ChemicalEntity_Disease},{ChemicalEntity},{Disease},{CROssBAR},"{Drugbank, Pubchem}",{DOID}
4,ChemicalEntity_Protein_Crossbar.csv,15201,{ChemicalEntity_Protein},{ChemicalEntity},{Protein},{CROssBAR},{Pubchem},{Uniprot_ID}
5,Chemical_Protein_Crossbar.csv,568,{ChemicalEntity_Protein},{ChemicalEntity},{Protein},{CROssBAR},{Pubchem},{Uniprot_ID}
6,Disease_Phenotype_Crossbar.csv,920,{Disease_Phenotype},{Disease},{Phenotype},{CROssBAR},{DOID},{HPO_ID}
7,Disease_Protein_1_Crossbar.csv,1229,{Disease_Protein},{Disease},{Protein},{CROssBAR},{DOID},{Uniprot_ID}
8,Disease_Protein_Crossbar.csv,702,{Disease:Kegg_Protein},{Disease:Kegg},{Protein},{CROssBAR},{DOID},{Uniprot_ID}
9,MolecularFunction_MolecularFunction_Crossbar.csv,13760,{MolecularFunction_MolecularFunction},{MolecularFunction},{MolecularFunction},{CROssBAR},{QuickGO},{QuickGO}



Total triples: 3,685,351
